# 09. Test a small style feature addition

Owner: Gtpatel02. Issue #9. Depends on completed #8.

## Goal

Test whether nine simple style features help the word C4 handoff. The hypothesis and adoption rule below are fixed and committed before any run.

## Hypothesis and rule (fixed before running)

**Hypothesis.** The word model lowercases text and its tokenizer drops punctuation, line breaks, digits as formatting, and sentence boundaries. In #8 the weakest validation bins were the top uppercase quartile (AUROC 0.865) and the shortest quartile (0.912). Adding nine scaled style features to the same word TF-IDF and the same logistic C4 will raise validation AUROC, and the largest bin gains will be in those two bins.

**Adoption rule.** Adopt only if all three hold:
1. Validation AUROC gain of at least .002.
2. Paired class-stratified bootstrap 95% interval for the gain (2,000 draws, seed 42) entirely above 0.
3. Every one of five training CV folds improves (fold gain > 0).

If any condition fails, the change is not adopted and the failed condition is shown. Subgroup changes are descriptive only and do not decide adoption.

**Prior note.** We ran a near-identical test in an earlier team repo, and it was inconclusive. The rule here comes from the issue, not from that result.

## Setup

Load the #8 analysis functions in helpers-only mode (this also loads notebook 02). Both arms share the split, word settings, classifier, seed, and folds. The only difference is the style block.

In [ ]:
import json, os, re, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from threadpoolctl import threadpool_limits
# Load #8 helpers without running #8. Notebook 08 also loads notebook 02.
os.environ["ERROR_ANALYSIS_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./08_error_analysis.ipynb")
NAME = "09_linear_style"
COMMAND = "uv run --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=900 notebooks/09_linear_style.ipynb"
STYLE_FEATURES = {
    "log_words": "log(1 + whitespace-split word count)",
    "line_breaks_per_word": "newline count / max(1, words)",
    "punct_per_char": "chars that are not alphanumeric or whitespace / max(1, chars)",
    "upper_per_letter": "uppercase letters / max(1, letters)",
    "digits_per_char": "digit chars / max(1, chars)",
    "log_words_per_sentence": "log(1 + words / max(1, runs of [.!?]+))",
    "type_token_ratio": "unique lowercase \\w+ tokens / max(1, tokens)",
    "nonascii_per_char": "chars with code point > 127 / max(1, chars)",
    "list_line_rate": "lines starting with -, *, bullet, or 1. / 1) then a space / max(1, newlines + 1)",
}
CONFIG = {
    "reference": "word TF-IDF logistic C4, same settings as the #4 and #5 handoff",
    "vectorizer": {"analyzer": "word", "ngram_range": [1, 2], "min_df": 2, "max_features": 50000,
                   "lowercase": True, "sublinear_tf": True, "norm": "l2", "dtype": "float32"},
    # L2 is the default penalty. Passing it explicitly is deprecated in scikit-learn 1.8.
    "classifier": {"C": 4.0, "solver": "liblinear", "max_iter": 2000, "random_state": SEED, "tol": 1e-4},
    "classifier_penalty": "l2 (scikit-learn default, not passed)",
    "style_features": STYLE_FEATURES,
    "style_scaling": "StandardScaler inside the pipeline, fit on each fit's training rows only",
    "cv": {"n_splits": 5, "shuffle": True, "random_state": SEED},
    "bootstrap": {"draws": 2000, "seed": SEED, "sampling": "paired class-stratified percentile 95%"},
    "adoption": {"min_validation_gain": 0.002, "ci_low_must_exceed": 0.0, "every_fold_improves": True},
    "reference_check": "reference validation AUROC must match the #5 handoff record within 1e-9",
    "threads": 1,
}
# Hash source cells only, so the record names the exact committed code, not outputs.
SOURCE_HASH = value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks/09_linear_style.ipynb").read_text())["cells"]])
CONFIG["notebook_source_sha256"] = SOURCE_HASH
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
START = time.perf_counter()

## Checks

Recheck the split and the #5 handoff record. Folds are built once and their ID hashes are saved, so both arms see exactly the same rows.

In [ ]:
parts = load_partitions()
train, validation = parts["train"], parts["validation"]
pointer = json.loads((ROOT / "artifacts/05_char_comparison/latest.json").read_text())
require(pointer["split_sha256"] == parts["split"]["split_sha256"], "Handoff used a different split")
require(file_hash(ROOT / pointer["result"]) == pointer["result_sha256"], "Handoff record changed")
handoff_record = json.loads((ROOT / pointer["result"]).read_text())
require(pointer["handoff"] == "word" and pointer["word_family"] == "logistic", "Handoff is not word logistic")
HANDOFF_AUROC = handoff_record["metrics"]["validation_auroc"]
folds = list(StratifiedKFold(**CONFIG["cv"]).split(train.text, train.label))
CONFIG["fold_id_sha256"] = [{"fit": value_hash(train.id.iloc[a].tolist()), "score": value_hash(train.id.iloc[b].tolist())}
                            for a, b in folds]
print(f"Train {len(train)}, validation {len(validation)}, folds {len(folds)}, handoff validation AUROC {HANDOFF_AUROC:.6f}")

## Method

Nine deterministic features per passage. Every denominator has a floor of 1, so empty text gives 0, not NaN. The scaler sits inside the pipeline, so it only ever sees the rows being fit.

The reference arm is word TF-IDF plus logistic C4. The style arm joins the same TF-IDF with the scaled style block and feeds the same classifier.

In [ ]:
SENTENCE_END = re.compile(r"[.!?]+")
LIST_LINE = re.compile(r"^\s*(?:[-*\u2022]|\d+[.)])\s", re.MULTILINE)
TOKEN = re.compile(r"\w+")

def style_features(texts):
    rows = []
    for text in texts:
        text = str(text)
        words = len(text.split())
        chars = max(1, len(text))
        letters = sum(c.isalpha() for c in text)
        tokens = TOKEN.findall(text.lower())
        rows.append([
            np.log1p(words),
            text.count("\n") / max(1, words),
            sum(not c.isalnum() and not c.isspace() for c in text) / chars,
            sum(c.isupper() for c in text) / max(1, letters),
            sum(c.isdigit() for c in text) / chars,
            np.log1p(words / max(1, len(SENTENCE_END.findall(text)))),
            len(set(tokens)) / max(1, len(tokens)),
            sum(ord(c) > 127 for c in text) / chars,
            len(LIST_LINE.findall(text)) / max(1, text.count("\n") + 1),
        ])
    return np.asarray(rows, dtype=np.float32)

def make_vectorizer():
    settings = dict(CONFIG["vectorizer"])
    settings["ngram_range"], settings["dtype"] = tuple(settings["ngram_range"]), np.float32
    return TfidfVectorizer(**settings)

def make_model(arm):
    classifier = LogisticRegression(**CONFIG["classifier"])
    if arm == "reference":
        return Pipeline([("tfidf", make_vectorizer()), ("classifier", classifier)])
    require(arm == "style", "Unknown arm")
    style = Pipeline([("measure", FunctionTransformer(style_features)), ("scale", StandardScaler())])
    return Pipeline([("features", FeatureUnion([("tfidf", make_vectorizer()), ("style", style)])),
                     ("classifier", classifier)])

def paired_gain_interval(labels, reference, treatment, draws, seed):
    labels, reference, treatment = (np.asarray(x) for x in (labels, reference, treatment))
    positives, negatives = np.flatnonzero(labels == 1), np.flatnonzero(labels == 0)
    rng = np.random.default_rng(seed)
    gains = []
    for _ in range(draws):
        # Same resampled rows for both arms, so the interval is for the paired gain.
        idx = np.concatenate([rng.choice(positives, len(positives)), rng.choice(negatives, len(negatives))])
        gains.append(roc_auc_score(labels[idx], treatment[idx]) - roc_auc_score(labels[idx], reference[idx]))
    low, high = np.quantile(gains, [.025, .975])
    return float(low), float(high), float(np.mean(np.asarray(gains) > 0))

In [ ]:
# Synthetic rows only. These test feature code, not the model.
toy = style_features(["Hello World.\n- one\n- two", "plain words here", "", "Ünïcode 42!"])
require(toy.shape == (4, len(STYLE_FEATURES)) and toy.dtype == np.float32, "Wrong feature shape or dtype")
first = dict(zip(STYLE_FEATURES, toy[0]))
require(abs(first["line_breaks_per_word"] - 2 / 6) < 1e-6, "Line break rate failed")
require(abs(first["upper_per_letter"] - 2 / 16) < 1e-6, "Uppercase rate failed")
require(abs(first["list_line_rate"] - 2 / 3) < 1e-6, "List line rate failed")
require(np.isfinite(toy).all() and (toy[2] == 0).all(), "Empty text must give zeros")
require(dict(zip(STYLE_FEATURES, toy[3]))["nonascii_per_char"] > 0, "Non-ASCII rate failed")
low, high, share = paired_gain_interval([0, 0, 1, 1], [.1, .6, .4, .9], [.1, .2, .8, .9], 50, SEED)
require(low <= high and 0 <= share <= 1, "Bootstrap interval failed")
print("Synthetic feature checks passed.")

## Run

Save the plan before any fit. Then five CV folds for both arms, the full 8,000-row fits, validation scores, the bootstrap, and subgroup tables. Convergence warnings stop the run instead of passing silently.

In [ ]:
plan_path = ROOT / "results" / NAME / f"{RUN_ID}-plan.json"
save_new_json(plan_path, {"config": CONFIG, "config_sha256": value_hash(CONFIG), "saved_before_fit": True,
                          "split_sha256": parts["split"]["split_sha256"]})
fold_rows, fit_seconds, cv_seconds, predictions = [], {}, {}, {}
with threadpool_limits(limits=1), warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    for arm in ["reference", "style"]:
        cv_start = time.perf_counter()
        for fold, (fit_idx, score_idx) in enumerate(folds):
            model = make_model(arm).fit(train.text.iloc[fit_idx], train.label.iloc[fit_idx])
            p = class1_probabilities(model, train.text.iloc[score_idx])
            fold_rows.append({"arm": arm, "fold": fold, "auroc": float(roc_auc_score(train.label.iloc[score_idx], p))})
        cv_seconds[arm] = time.perf_counter() - cv_start
        fit_start = time.perf_counter()
        model = make_model(arm).fit(train.text, train.label)
        fit_seconds[arm] = time.perf_counter() - fit_start
        predictions[arm] = {name: pd.DataFrame({"id": frame.id.to_numpy(), "p_ai": class1_probabilities(model, frame.text)})
                            for name, frame in [("train", train), ("validation", validation)]}
folds_table = pd.DataFrame(fold_rows).pivot(index="fold", columns="arm", values="auroc")
folds_table["gain"] = folds_table["style"] - folds_table["reference"]
scores = {arm: {name: evaluate_predictions(frame[["id", "label"]], predictions[arm][name])
                for name, frame in [("train", train), ("validation", validation)]} for arm in predictions}
require(abs(scores["reference"]["validation"]["auroc"] - HANDOFF_AUROC) <= 1e-9, "Reference arm does not reproduce the #5 handoff")
paired = validation[["id", "label"]].merge(predictions["reference"]["validation"], on="id", validate="one_to_one").merge(
    predictions["style"]["validation"], on="id", suffixes=("_reference", "_style"), validate="one_to_one")
boot_start = time.perf_counter()
ci_low, ci_high, share_positive = paired_gain_interval(paired.label, paired.p_ai_reference, paired.p_ai_style,
                                                       CONFIG["bootstrap"]["draws"], SEED)
bootstrap_seconds = time.perf_counter() - boot_start
gain = scores["style"]["validation"]["auroc"] - scores["reference"]["validation"]["auroc"]
rule = CONFIG["adoption"]
conditions = [
    {"condition": "validation gain >= .002", "value": gain, "passed": gain >= rule["min_validation_gain"]},
    {"condition": "bootstrap CI low > 0", "value": ci_low, "passed": ci_low > rule["ci_low_must_exceed"]},
    {"condition": "every CV fold improves", "value": float(folds_table["gain"].min()), "passed": bool((folds_table["gain"] > 0).all())},
]
adopted = all(c["passed"] for c in conditions)
measures = {"train": measure(train), "validation": measure(validation)}
edges = {column: quartile_edges(measures["train"][column]) for column in ["words", "line_breaks", "upper_per_letter", "punct_per_char"]}
subgroups = subgroup_table(validation, predictions["reference"]["validation"], measures["validation"], edges, "reference", draws=0).merge(
    subgroup_table(validation, predictions["style"]["validation"], measures["validation"], edges, "style", draws=0)[["measure", "bin", "auroc"]],
    on=["measure", "bin"], suffixes=("_reference", "_style"))
subgroups["gain"] = subgroups["auroc_style"] - subgroups["auroc_reference"]
run_seconds = time.perf_counter() - START

## Results

The bootstrap interval conditions on the two fitted models and the reused 2,000 validation rows. It does not cover training variation. Validation was also used to pick the #4 handoff and to find the weak bins in #8, so the gain here is not an unbiased estimate.

In [ ]:
display(folds_table.round(6))
display(pd.DataFrame([{"arm": arm, "train_auroc": scores[arm]["train"]["auroc"],
                       "validation_auroc": scores[arm]["validation"]["auroc"],
                       "validation_accuracy_at_0_5": scores[arm]["validation"]["accuracy_at_0_5"],
                       "cv_mean": folds_table[arm].mean(), "cv_std": folds_table[arm].std(ddof=0),
                       "cv_seconds": cv_seconds[arm], "fit_seconds": fit_seconds[arm]} for arm in ["reference", "style"]]))
print(f"Validation gain {gain:+.6f}; paired 95% CI [{ci_low:+.6f}, {ci_high:+.6f}]; share of draws above 0: {share_positive:.3f}")
display(pd.DataFrame(conditions))
print("ADOPTED" if adopted else "NOT ADOPTED: " + "; ".join(c["condition"] for c in conditions if not c["passed"]))
display(subgroups[["measure", "bin", "n", "human", "ai", "auroc_reference", "auroc_style", "gain"]].round(4))
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(folds_table.index, folds_table["gain"])
ax.axhline(0, color="black", linewidth=.8)
ax.set(xlabel="training CV fold", ylabel="style minus reference AUROC", title="Fold gains")
fig.tight_layout()
plot_path = ROOT / "results" / NAME / f"{RUN_ID}-fold-gains.png"
fig.savefig(plot_path, dpi=140)
plt.show()
print(f"Bootstrap {bootstrap_seconds:.3f}s; whole run {run_seconds:.3f}s")

## Takeaway

- Tried: nine scaled style features added to the same word TF-IDF and logistic C4, with the same split, five folds, and seed. Rule fixed before running.
- Found: validation rose 0.941252 to 0.944372 (+0.003120), but the paired 95% CI is -0.002509 to +0.009001 and fold 4 dropped by 0.000356. Two of three conditions fail, so **not adopted**. The top uppercase bin gained +0.0301 as predicted, but the shortest bin lost -0.0253, against the hypothesis. Fitting takes about twice as long (3.5 s vs 1.5 s).
- Next: keep word C4 as the handoff. #10 tests word plus style again inside a wider grid; watch the short-text bin there.

Guess, not tested: per-text rates are noisy on short passages, which could explain the short-text loss.

## Save

Validation probabilities for both arms stay under ignored `artifacts/09_linear_style/`. The safe record keeps fold scores, both arms, the bootstrap, each adoption condition, subgroup changes, and timings. Models are not saved; #10 refits its own word plus style candidate.

In [ ]:
private = ROOT / "artifacts" / NAME / RUN_ID
private.mkdir(parents=True, exist_ok=False)
artifacts = [plan_path, plot_path]
for arm in ["reference", "style"]:
    path = private / f"{arm}_validation.csv"
    predictions[arm]["validation"].to_csv(path, index=False)
    artifacts.append(path)
metrics = {"validation_auroc": scores["style"]["validation"]["auroc"],
           "reference_validation_auroc": scores["reference"]["validation"]["auroc"],
           "arms": {arm: {"train_auroc": scores[arm]["train"]["auroc"], "validation_auroc": scores[arm]["validation"]["auroc"],
                          "validation_accuracy_at_0_5": scores[arm]["validation"]["accuracy_at_0_5"],
                          "cv_fold_auroc": folds_table[arm].tolist(), "cv_mean": float(folds_table[arm].mean()),
                          "cv_std": float(folds_table[arm].std(ddof=0)), "cv_seconds": cv_seconds[arm],
                          "fit_seconds": fit_seconds[arm]} for arm in ["reference", "style"]},
           "validation_gain": gain, "bootstrap_ci": [ci_low, ci_high], "bootstrap_share_positive": share_positive,
           "fold_gains": folds_table["gain"].tolist(), "adoption_conditions": conditions, "adopted": adopted,
           "subgroups": json.loads(subgroups.to_json(orient="records")), "bootstrap_seconds": bootstrap_seconds}
CONFIG["extra_versions"] = {name: importlib.metadata.version(name) for name in ["matplotlib", "threadpoolctl"]}
record_path = record_run(NAME, COMMAND, CONFIG, SEED, len(train), len(validation), metrics,
                         time.perf_counter() - START, artifacts=artifacts)
print(f"Saved {record_path.relative_to(ROOT)}")